# Laya-1.2 FP16 → packed ternary G128 → cascaded FP16 GEMM

**Model:** [vtava/Laya-1.2-speedup-standalone-fp16](https://huggingface.co/vtava/Laya-1.2-speedup-standalone-fp16)  
**Source:** [vtavakkoli/TinyCeNN-LM](https://github.com/vtavakkoli/TinyCeNN-LM)

This is a **new, standalone, layer-by-layer experimental notebook** that quantizes the real Laya-1.2 model and compares three ways of executing the *same packed ternary weights*.

1. **Packed → Triton:** unpack each 128-weight group in a fused CUDA kernel, perform FP16 tensor-core dot products, accumulate in FP32.
2. **Packed → torch cascade:** decompress one group at a time, run normal FP16 GEMM and add partial results in FP32. Reference and performance control; not expected to be fast.
3. **Packed → cached FP16:** decompress each converted matrix once and use cuBLAS. Fast reference, but discards the runtime GPU-memory saving.
4. **Original FP16:** unmodified teacher baseline.

Weights are **{-1,0,+1} with independent FP16 scales per output-channel group of 128 input weights**, stored as **2-bit codes + scale overhead ≈ 2.125 bits/weight**. This is *not* 1.585-bit storage, a trained BitNet, or the proprietary Bonsai algorithm.

Experiment selection is made using a separate train-split development set. The test set is never used during selection. **No GPU results are claimed until Colab runs.** Source model and existing GitHub notebooks remain unchanged. CUDA T4 or L4 recommended.

**Start with `MAX_LAYERS=6`**, compare backends and use the JSON results to determine whether it is worth quantizing further. This is primarily a GPU-kernel/format experiment, not a QAT-training run.


In [ ]:
#@title 1. Install exact runtime dependencies
%pip -q install "transformers==4.57.6" "huggingface_hub>=0.25,<1" "safetensors>=0.4" "datasets>=2.20,<5" "triton>=3.0,<4" pandas matplotlib


In [ ]:
#@title 2. Configure experiment and download original FP16 checkpoint
import gc, json, os, time, math, random, copy, importlib.util, inspect, sys, shutil, zipfile
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch import nn
from torch.nn import functional as F
from huggingface_hub import snapshot_download
from IPython.display import display

MODEL_REPO = "vtava/Laya-1.2-speedup-standalone-fp16" #@param {type:"string"}
MAX_LAYERS = 6 #@param {type:"integer"}
FIRST_LAYER = 0 #@param {type:"integer"}
DEV_CASES = 130 #@param {type:"integer"}
DEV_DECISIONS = 96 #@param {type:"integer"}
TEST_CASES = 70 #@param {type:"integer"}
TEST_DECISIONS = 128 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}
MIN_LINEAR_ELEMENTS = 4096 #@param {type:"integer"}
MIN_DECISION_AGREEMENT = 0.97 #@param {type:"number"}
MAX_MEAN_JS = 0.015 #@param {type:"number"}
BENCH_REPEATS = 10 #@param {type:"integer"}
CASCADE_FALLBACK_MARGIN = 0.12 #@param {type:"number"}
SAVE_PACKED_DELTA = True #@param {type:"boolean"}
SEED=2026
OUT=Path("/content/Laya_FP16_Ternary_Cascade")
OUT.mkdir(exist_ok=True,parents=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Please select T4/L4/A100 GPU in Colab Runtime settings."
assert torch.cuda.get_device_capability()[0] >= 7, "FP16 tensor-core CUDA GPU required"
try:
    from google.colab import userdata
    TOKEN=userdata.get("HF_TOKEN")
except Exception:
    TOKEN=os.environ.get("HF_TOKEN")
if not TOKEN:
    print("HF_TOKEN not supplied; trying public access. Add Colab secret HF_TOKEN if model is private.")
ROOT=Path(snapshot_download(repo_id=MODEL_REPO,token=TOKEN))
print("Checkpoint:",ROOT)
print("GPU:",torch.cuda.get_device_name(0),"| checkpoint files:", sorted(p.name for p in ROOT.iterdir()))
assert (ROOT/"model.safetensors").is_file(), "Expected a complete standalone checkpoint."
assert (ROOT/"config.json").is_file(), "Missing standalone config.json"
CFG=json.loads((ROOT/"config.json").read_text())
print("Checkpoint configuration:",CFG)


In [ ]:
#@title 3. Load exact standalone Laya source, not a generic Transformers CausalLM
def import_py(path, name):
    spec=importlib.util.spec_from_file_location(name,str(path))
    assert spec is not None and spec.loader is not None
    module=importlib.util.module_from_spec(spec)
    sys.modules[name]=module
    spec.loader.exec_module(module)
    return module

modules=[]
for filename in ("modeling_laya_standalone.py","load_model.py","runtime.py"):
    path=ROOT/filename
    if path.is_file():
        modules.append((filename,import_py(path,"_laya_standalone_"+filename[:-3])))
loaders=[(filename,m) for filename,m in modules if callable(getattr(m,"load_model",None))]
assert loaders, "Standalone load_model() not found; examine downloaded files and adapt loader."
loader_name,source=loaders[0]
kwargs=dict(device="cpu",local_files_only=True)
accepted=set(inspect.signature(source.load_model).parameters)
kwargs={k:v for k,v in kwargs.items() if k in accepted}
reference,tokenizer=source.load_model(ROOT,**kwargs)
reference=reference.to("cuda",dtype=torch.float16).eval().requires_grad_(False)
for _,m in modules:
    if all(hasattr(m,k) for k in ("build_sequence","normalize_question")):
        runtime=m
        break
else:
    raise RuntimeError("Cannot find original Laya build_sequence()/normalize_question().")
print("Loader:",loader_name,"| model class:",type(reference).__name__)
print("Encoder:",type(reference.encoder).__name__)
print("Model params:",f"{sum(p.numel() for p in reference.parameters()):,}")
assert hasattr(reference.encoder,"layers"), "Missing encoder.layers in this standalone checkpoint."
N_LAYERS=len(reference.encoder.layers)
print("Encoder layers:",N_LAYERS)
assert 0 <= FIRST_LAYER < N_LAYERS and MAX_LAYERS>0


In [ ]:
#@title 4. Load pinned ternary runtime and verify CPU packing + GPU math
from urllib.request import urlopen
BACKEND_REVISION="ed1ede5f4fc6430b78fab6b66e87d4f4053f2469"
BACKEND_URL=("https://raw.githubusercontent.com/vtavakkoli/TinyCeNN-LM/"+
             BACKEND_REVISION+"/src/tinycenn_lm/laya_fp16_ternary_cascade.py")
codefile=Path("/content/laya_fp16_ternary_cascade.py")
with urlopen(BACKEND_URL,timeout=90) as r:
    codefile.write_bytes(r.read())
q=import_py(codefile,"laya_fp16_ternary_cascade")
assert q.TRITON_AVAILABLE, "Triton missing from Colab runtime"
# Actual correctness check against decoded dense FP16 with SAME quantized weights
probe=nn.Linear(768,768,bias=True,device="cuda",dtype=torch.float16)
packed=q.TernaryCascadeLinear.from_linear(probe,backend="triton")
raw_codes=q.unpack_ternary(packed.packed_weight,packed.kpad)
assert torch.equal(q.pack_ternary(raw_codes),packed.packed_weight)
px=torch.randn(192,768,device="cuda",dtype=torch.float16)
with torch.inference_mode():
    target=F.linear(px,packed.dense_weight(),packed.bias).float()
    kernel=packed(px).float()
    packed.set_backend("torch_cascade")
    cascade=packed(px).float()
    packed.set_backend("cached_fp16")
    cache=packed(px).float()
for title,actual in (("Triton",kernel),("Torch cascade",cascade),("Cached FP16",cache)):
    delta=(actual-target).abs()
    print(title,"max absolute error:",round(float(delta.max()),5),
          "mean absolute error:",round(float(delta.mean()),6))
    torch.testing.assert_close(actual,target,atol=0.06,rtol=0.02)
packed.set_backend("triton")
print("Packed ternary CPU/GPU and FP16 cascaded arithmetic: PASS")
del probe, packed, px, target, kernel, cascade, cache, raw_codes
gc.collect();torch.cuda.empty_cache()


In [ ]:
#@title 5. Build independent real typed-decision development/test examples
from datasets import load_dataset
def as_obj(v):
    if isinstance(v,str):
        try:return json.loads(v)
        except Exception:return v
    return v

def parse_items(rows,limit):
    items=[]
    for row in rows:
        state=as_obj(row.get("state"))
        questions=as_obj(row.get("questions"))
        if not isinstance(questions,dict):continue
        for question in questions.values():
            try:
                question=runtime.normalize_question(as_obj(question))
                ids,markers=runtime.build_sequence(tokenizer,state,question,
                    max_len=MAX_LEN,head_max_len=192,truncate_left=isinstance(state,list))
                if len(markers)<2:continue
                items.append(dict(ids=ids,markers=markers,qtype=runtime.QTYPES[question["t"]]))
            except (ValueError,TypeError,KeyError,IndexError):continue
            if len(items)>=limit:return items
    return items

train=load_dataset("LocalLLaMA/typed-decisions","all",split=f"train[:{DEV_CASES}]")
test=load_dataset("LocalLLaMA/typed-decisions","all",split=f"test[:{TEST_CASES}]")
dev_items=parse_items(train,DEV_DECISIONS)
test_items=parse_items(test,TEST_DECISIONS)
assert len(dev_items)==DEV_DECISIONS and len(test_items)==TEST_DECISIONS, (
    f"Not enough valid decisions: dev={len(dev_items)}, test={len(test_items)}")
print("Real disjoint split decisions:",len(dev_items),"development and",len(test_items),"test")
def batches(items,size=4,seq_length=None):
    result=[]
    for index in range(0,len(items),size):
        chosen=items[index:index+size]
        batch=runtime.collate_items(chosen,int(tokenizer.pad_token_id))
        if seq_length is not None:
            # Only add right-padding: do not truncate decisions or marker positions.
            needed=batch["input_ids"].shape[1]
            if needed>seq_length:
                raise ValueError(f"Example needs {needed} tokens, above benchmark length {seq_length}")
            diff=seq_length-needed
            if diff:
                batch["input_ids"]=F.pad(batch["input_ids"],(0,diff),
                    value=int(tokenizer.pad_token_id))
                batch["attention_mask"]=F.pad(batch["attention_mask"],(0,diff),value=0)
        result.append({key:v.to("cuda") for key,v in batch.items()})
    return result

DEV_BATCHES=batches(dev_items)
TEST_BATCHES=batches(test_items)
print("Dev/test batch lengths:",
    sorted(set(v["input_ids"].shape[1] for v in DEV_BATCHES)),
    sorted(set(v["input_ids"].shape[1] for v in TEST_BATCHES)))


In [ ]:
#@title 6. Record FP16 reference logits (no effect from subsequent quantization)
@torch.inference_mode()
def forward(m,b):
    with torch.autocast("cuda",dtype=torch.float16):
        out=m(**b)
    return out[0] if isinstance(out,(list,tuple)) else out

def reference_logits(model,batch_list):
    return [forward(model,b).detach().float().cpu() for b in batch_list]

DEV_TEACHER=reference_logits(reference,DEV_BATCHES)
TEST_TEACHER=reference_logits(reference,TEST_BATCHES)

def compare(candidate,batch_list,reference_logits_list):
    matches=n=0
    js_total=0.0
    max_error=0.0
    with torch.inference_mode():
        for b,reference_logit in zip(batch_list,reference_logits_list):
            y=forward(candidate,b).float().cpu()
            r=reference_logit
            marker=b["marker_mask"].cpu()
            y=y.masked_fill(~marker,-1e4)
            r=r.masked_fill(~marker,-1e4)
            p=torch.softmax(y,dim=-1).clamp_min(1e-9)
            t=torch.softmax(r,dim=-1).clamp_min(1e-9)
            mid=0.5*(p+t)
            js=0.5*(p*(p/mid).log()).sum(-1)+0.5*(t*(t/mid).log()).sum(-1)
            js_total+=float(js.sum())
            max_error=max(max_error,float((y-r).abs().max()))
            matches+=int((p.argmax(-1)==t.argmax(-1)).sum())
            n+=len(y)
    return dict(n=n,agreement=matches/n,mean_js=js_total/n,max_logit_error=max_error)

print("Reference self-consistency:",compare(reference,DEV_BATCHES,DEV_TEACHER))


In [ ]:
#@title 7. Microbenchmark dense FP16 vs three ternary calculation methods
@torch.inference_mode()
def timer(fn,repeats=BENCH_REPEATS,warmup=4):
    for _ in range(warmup):fn()
    torch.cuda.synchronize()
    events=[]
    for _ in range(repeats):
        t0=torch.cuda.Event(enable_timing=True)
        t1=torch.cuda.Event(enable_timing=True)
        t0.record();fn();t1.record()
        torch.cuda.synchronize()
        events.append(t0.elapsed_time(t1))
    return float(np.median(events))

micro=[]
for input_features,output_features,mrows in [(768,768,128),(768,3072,512),(3072,768,512)]:
    linear=nn.Linear(input_features,output_features,device="cuda",dtype=torch.float16).eval()
    x=torch.randn(mrows,input_features,device="cuda",dtype=torch.float16)
    module=q.TernaryCascadeLinear.from_linear(linear)
    variants={"Original FP16":lambda:linear(x)}
    for backend in ("triton","torch_cascade","cached_fp16"):
        module.set_backend(backend)
        # warm up once so cached-FP16 backend is truly cached for the timing.
        _=module(x)
        variants[backend]=timer(lambda:module(x))
    original=timer(variants["Original FP16"])
    micro.append(dict(input=input_features,output=output_features,M=mrows,
        fp16_ms=original,triton_ms=variants["triton"],
        torch_cascade_ms=variants["torch_cascade"],
        cached_fp16_ms=variants["cached_fp16"],
        original_vs_triton_speedup=original/variants["triton"]))
    del linear,module,x
    gc.collect();torch.cuda.empty_cache()
MICRO=pd.DataFrame(micro)
display(MICRO)
MICRO.to_csv(OUT/"microbenchmark.csv",index=False)


In [ ]:
#@title 8. Layer-by-layer quantization with rollback using development decisions ONLY
candidate=copy.deepcopy(reference).eval().requires_grad_(False)
original_storage=q.state_bytes(reference)
all_layers=list(range(FIRST_LAYER,min(N_LAYERS,FIRST_LAYER+MAX_LAYERS)))
rows=[];accepted=[]
for layer_index in all_layers:
    prefix=f"encoder.layers.{layer_index}."
    projections=[(name,module) for name,module in candidate.named_modules()
        if name.startswith(prefix) and isinstance(module,nn.Linear)
        and module.weight.numel()>=MIN_LINEAR_ELEMENTS]
    print(f"\nLayer {layer_index:02d}: {len(projections)} candidate FP16 projections")
    # Largest first; committing only if original-model fidelity stays within gate.
    for name,original_linear in sorted(projections,key=lambda p:-p[1].weight.numel()):
        try:
            quant=q.TernaryCascadeLinear.from_linear(original_linear,backend="triton")
            q.replace(candidate,name,quant)
            score=compare(candidate,DEV_BATCHES,DEV_TEACHER)
            saved=q.state_bytes(original_linear)-q.state_bytes(quant)
            passed=(score["agreement"]>=MIN_DECISION_AGREEMENT
                    and score["mean_js"]<=MAX_MEAN_JS
                    and saved>0)
            if not passed:q.replace(candidate,name,original_linear)
            else:accepted.append(name)
            rec={"layer":layer_index,"module":name,"accepted":passed,
                 "saved_mib":saved/(1024**2),**score}
            rows.append(rec)
            print(f'  {name.rsplit(".",2)[-2:]}: agreement={score["agreement"]:.3%}, '
                  f'JS={score["mean_js"]:.5f}, save={rec["saved_mib"]:.2f}MiB, '
                  f'{"KEEP" if passed else "REVERT"}')
        except Exception as e:
            q.replace(candidate,name,original_linear)
            rows.append(dict(layer=layer_index,module=name,accepted=False,error=repr(e)))
            print("  Error — reverted:",name,repr(e))
    pd.DataFrame(rows).to_csv(OUT/"layerwise_trials.csv",index=False)
    (OUT/"accepted_modules.json").write_text(json.dumps(accepted,indent=2))
trials=pd.DataFrame(rows)
print("Accepted",len(accepted),"of",len(trials),"modules")
print("Packed state size MiB:",q.state_bytes(candidate)/(1024**2),
      "vs FP16:",original_storage/(1024**2))
display(trials)


In [ ]:
#@title 9. Independent accuracy: test set was not used to select layers
packed_dev=compare(candidate,DEV_BATCHES,DEV_TEACHER)
packed_test=compare(candidate,TEST_BATCHES,TEST_TEACHER)
print("DEVELOPMENT:",packed_dev)
print("INDEPENDENT TEST:",packed_test)
if packed_test["agreement"]<MIN_DECISION_AGREEMENT or packed_test["mean_js"]>MAX_MEAN_JS:
    print("WARNING: Packed checkpoint FAILED independent quality gate; do not publish as validated.")


In [ ]:
#@title 10. End-to-end benchmark: original FP16 vs packed vs FP16 cached and explicit cascade
def set_backend_all(m,backend):
    for sub in m.modules():
        if isinstance(sub,q.TernaryCascadeLinear):
            sub.set_backend(backend)

# Use real examples and right-pad them to requested fixed lengths.
BENCH_CASES={}
for batch_size in (1,4):
    example=test_items[:batch_size]
    min_tokens=max(len(z["ids"]) for z in example)
    for width in (128,256,512):
        if width>=min_tokens and width<=MAX_LEN:
            BENCH_CASES[f"B{batch_size}_T{width}"]=batches(
                example,size=batch_size,seq_length=width)[0]
assert BENCH_CASES,"No valid benchmark sequences"
results=[]
for label,b in BENCH_CASES.items():
    baseline=timer(lambda:forward(reference,b))
    for backend in ("triton","cached_fp16","torch_cascade"):
        set_backend_all(candidate,backend)
        _=forward(candidate,b)
        measured=timer(lambda:forward(candidate,b))
        results.append(dict(shape=label,backend=backend,original_fp16_ms=baseline,
            compressed_ms=measured,speedup=baseline/measured,
            packed_state_mib=q.state_bytes(candidate)/(1024**2),
            resident_weight_mib=q.runtime_bytes(candidate)/(1024**2)))
    print(label,"FP16:",round(baseline,2),"ms")
set_backend_all(candidate,"triton")
BENCH=pd.DataFrame(results)
display(BENCH)
BENCH.to_csv(OUT/"end_to_end_bench.csv",index=False)


In [ ]:
#@title 11. Optional confidence cascade: fall back to original FP16 on uncertain decisions
@torch.inference_mode()
def cascade_with_fallback(batch,margin=CASCADE_FALLBACK_MARGIN):
    logits=forward(candidate,batch).float()
    masked=logits.masked_fill(~batch["marker_mask"],-1e4)
    top=masked.topk(k=min(2,masked.shape[-1]),dim=-1).values
    if top.shape[-1]<2:
        return logits,0
    use_original=(top[:,0]-top[:,1])<margin
    count=int(use_original.sum())
    if count:
        subset={key:value[use_original] for key,value in batch.items()}
        fallback=forward(reference,subset).float()
        logits[use_original]=fallback
    return logits,count

fallback_rows=[]
for label,b in BENCH_CASES.items():
    elapsed=timer(lambda:cascade_with_fallback(b))
    _,count=cascade_with_fallback(b)
    ref=BENCH[(BENCH["shape"]==label)&(BENCH["backend"]=="triton")].iloc[0]
    fallback_rows.append(dict(shape=label,margin=CASCADE_FALLBACK_MARGIN,
        fallback_items=count,total_items=len(b["input_ids"]),cascade_ms=elapsed,
        original_fp16_ms=ref.original_fp16_ms,
        speedup_vs_original=ref.original_fp16_ms/elapsed))
FALLBACK=pd.DataFrame(fallback_rows)
display(FALLBACK)
print("Caution: the original reference model is kept resident in VRAM for fallbacks.")
FALLBACK.to_csv(OUT/"confidence_cascade_bench.csv",index=False)


In [ ]:
#@title 12. Export evidence and optionally save an experimental packed-weights delta
from safetensors.torch import save_file
report={
 "source":MODEL_REPO,
 "algorithm":"post_training_ternary_g128_packed2_fp16_tensorcore_fp32_cascade",
 "note":"Experiment, not trained BitNet/Bonsai; source checkpoint unchanged",
 "checkpoint_metadata":CFG,
 "gpu":torch.cuda.get_device_name(0),
 "layer_range":all_layers,
 "accepted":accepted,
 "original_weight_mib":original_storage/(1024**2),
 "packed_state_mib":q.state_bytes(candidate)/(1024**2),
 "selected_model_quality":packed_dev,
 "independent_test_quality":packed_test,
 "quality_gate_test_pass":bool(packed_test["agreement"]>=MIN_DECISION_AGREEMENT
                              and packed_test["mean_js"]<=MAX_MEAN_JS),
 "runtime_backends":["triton","torch_cascade","cached_fp16"],
 "fallback_margin":CASCADE_FALLBACK_MARGIN,
 "scope":"compressed state stats exclude keeping a separate full-FP16 teacher for fallback"
}
(OUT/"summary.json").write_text(json.dumps(report,indent=2,default=str))
if SAVE_PACKED_DELTA and accepted:
    quant_state={}
    for name in accepted:
        m=candidate.get_submodule(name)
        for key,value in m.state_dict().items():
            quant_state[f"{name}.{key}"]=value.detach().contiguous().cpu()
    save_file(quant_state,str(OUT/"ternary_delta.safetensors"))
    (OUT/"ternary_delta_meta.json").write_text(json.dumps({
        "source":MODEL_REPO,"module_names":accepted,
        "group_size":128,"encoding":"2bit codes: -1=0, 0=1, +1=2",
        "warning":"Delta requires exact original FP16 checkpoint to reload"
    },indent=2))
print(json.dumps({k:report[k] for k in ("original_weight_mib","packed_state_mib",
      "selected_model_quality","independent_test_quality","quality_gate_test_pass")},indent=2))
files_to_zip=[p for p in OUT.iterdir() if p.is_file()]
archive=OUT/"Laya_FP16_Ternary_Cascade_Results.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for path in files_to_zip:z.write(path,path.name)
print("Results ZIP:",archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    pass


## Interpreting the results

- **Speedup > 1:** faster than original FP16; **speedup < 1:** slower. Report actual measured times, not theoretical compute savings.
- **Triton faster than torch cascade:** the fused unpack kernel reduced overhead.
- **Cached FP16 faster than Triton:** the packed kernel is still the bottleneck; cached FP16 trades away the resident-memory savings.
- **Large quality loss:** one-shot ternary PTQ is too aggressive for those projections. Add layerwise reconstruction training / mixed INT4 only after a competitive GPU kernel is demonstrated.
- **Confidence fallback:** provides no automatic performance advantage. The notebook measures fallback counts and added latency, including cases where it is slower; the FP16 teacher must be loaded.
- **Bit width:** G128 gives 2.125 bits/weight before biases, alignment, and non-quantized layers; it is *not* genuine 1.58-bit physical storage.
- **No model overwrite:** this notebook never publishes, alters, or deletes the source Hugging Face checkpoint.

Production promotion would require strict validation on additional held-out tasks, a checkpoint reload-parity test, a faster dedicated kernel, and memory profiling without the FP16 teacher held resident.
